In [1]:
import pandas as pd

dataset = pd.read_csv("../data/auto_mpg.csv")
dataset = dataset.dropna()
dataset["Origin"] = dataset["Origin"].map({1: "USA", 2: "Europe", 3: "Japan"})
dataset.head()

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,USA
1,15.0,8,350.0,165.0,3693.0,11.5,70,USA
2,18.0,8,318.0,150.0,3436.0,11.0,70,USA
3,16.0,8,304.0,150.0,3433.0,12.0,70,USA
4,17.0,8,302.0,140.0,3449.0,10.5,70,USA


In [2]:
y = dataset.pop("MPG")
X = dataset

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
)
X_train.shape, X_test.shape

((294, 7), (98, 7))

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["Cylinders", "Displacement", "Horsepower", "Weight", "Acceleration", "Model Year"]
categorical_features = ["Origin"]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(), categorical_features),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("selectkbest", SelectKBest(score_func=f_regression)),
        ("regressor", LinearRegression()),
    ]
)

estimator = GridSearchCV(
    pipeline,
    param_grid={"selectkbest__k": list(range(1, 10))},
    cv=5,
    scoring="neg_mean_squared_error",
)
estimator.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['Cylinders',
                                                                          'Displacement',
                                                                          'Horsepower',
                                                                          'Weight',
                                                                          'Acceleration',
                                                                          'Model '
                                                                          'Year']),
                                                                        ('cat',
                                                                         OneHotEncoder(),
                                                                         ['Origin'])])),
                                       ('selectkbest',
                                        SelectKBest(score_func=<function f_regression at 0x0000016A1851D8A0>)),
                                       ('regressor', LinearRegression())]),
             param_grid={'selectkbest__k': [1, 2, 3, 4, 5, 6, 7, 8, 9]},
             scoring='neg_mean_squared_error')

In [4]:
results = pd.DataFrame(estimator.cv_results_)
results[["param_selectkbest__k", "mean_test_score"]]

,param_selectkbest__k,mean_test_score
0,1,-19.626008
1,2,-18.975843
2,3,-19.535759
3,4,-18.985050
4,5,-15.279822
5,6,-11.930599
6,7,-11.962486
7,8,-12.183604
8,9,-12.183604


In [5]:
best = estimator.best_estimator_
feature_names = best.named_steps["preprocessor"].get_feature_names_out()
selected = best.named_steps["selectkbest"].get_support()
estimator.best_params_, list(feature_names[selected])

({'selectkbest__k': 6},
 ['num__Cylinders',
  'num__Displacement',
  'num__Horsepower',
  'num__Weight',
  'num__Model Year',
  'cat__Origin_USA'])

In [6]:
from sklearn.metrics import mean_squared_error

mean_squared_error(y_train, estimator.predict(X_train)), mean_squared_error(y_test, estimator.predict(X_test))

(11.004326851457654, 10.270485534158666)

In [7]:
import os
import pickle

os.makedirs("../submission", exist_ok=True)

with open("../submission/estimator.pkl", "wb") as file:
    pickle.dump(estimator, file)